
# vLLM Source-Level Project — Day 1: Foundation

**Today's goal:** Environment → Run → Serve → Measure → Locate source.

You are **not** optimizing vLLM today. By the end of the notebook you should be able to:

1. Confirm the Colab T4 environment.
2. Install and record the exact vLLM/PyTorch/CUDA environment.
3. Run offline inference through `vllm.LLM`.
4. Inspect `RequestOutput`.
5. Run a very small latency experiment.
6. Start the OpenAI-compatible vLLM server.
7. Send one request through the HTTP serving path.
8. Record observations for Day 2.

### Rules
- Cells marked **TODO — YOU WRITE THIS** are intentionally incomplete.
- Try not to look up the final implementation until you have attempted it.
- Do not optimize anything yet.
- Use a **T4 runtime**: Runtime → Change runtime type → T4 GPU.


## 0. Confirm the GPU

In [1]:

!nvidia-smi


Fri Sep 18 10:26:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


### Checkpoint 0
Before continuing, answer in your notes:

- GPU model:
- Total VRAM:
- Driver version:
- CUDA version reported by `nvidia-smi`:


## 1. Install vLLM and helper packages

In [2]:
!pip uninstall -y torchaudio
!pip install -U torchaudio==2.11.0+cu130 \
  --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.5 MB/s eta 0:00:00


In [3]:

# vLLM changes quickly. For Day 1 we use the released package and record
# the exact installed version below so future benchmarks are reproducible.
%pip install -q -U vllm openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43


> If Colab asks you to restart the runtime after installation, restart it and then rerun from Cell 0.


## 2. Record the experiment environment

In [15]:

import sys
import torch
import vllm
import time
import statistics

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("vLLM   :", vllm.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GiB")


Python : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.13.0+cu130
Torch CUDA build: 13.0
vLLM   : 0.29.0
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GiB


In [ ]:
import os
os.kill(os.getpid(), 9)


### Experiment configuration

We intentionally use a small, ungated model so a T4 is sufficient.

For Day 1:

```text
MODEL = Qwen/Qwen2.5-1.5B-Instruct
dtype = float16
max_model_len = 4096
gpu_memory_utilization = 0.70
```

Why `float16`? T4 is a Turing GPU and FP16 is the safe choice for this exercise.


## 3. Offline inference — first core exercise

In [2]:
import torch
import torchaudio

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("torchaudio:", torchaudio.__version__)

torch: 2.13.0+cu130
torch CUDA: 13.0
torchaudio: 2.11.0+cu130


In [26]:
%%writefile /content/test_vllm.py
from vllm import LLM, SamplingParams
import time
import statistics

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70

output_path = "/content/request_output_inspection.txt"

prompts = [
    "Explain in one sentence what a KV cache is in LLM inference.",
    "Why can continuous batching improve GPU utilization?",
]

# ============================================================
# TODO — YOU WRITE THIS
#
# Task A: Construct SamplingParams.
# Requirements:
#   - temperature = 0.0
#   - max_tokens = 64
#
# sampling_params = ...
# ============================================================

sampling_params = SamplingParams(temperature=0.0,
                                 max_tokens=64)

# ============================================================
# TODO — YOU WRITE THIS
#
# Task B: Construct the vLLM LLM object.
#
# Requirements:
#   model=MODEL
#   dtype="float16"
#   max_model_len=MAX_MODEL_LEN
#   gpu_memory_utilization=GPU_MEMORY_UTILIZATION
#
# llm = ...
# ============================================================

llm = LLM(model = MODEL,
          dtype = "float16",
          max_model_len = MAX_MODEL_LEN,
          gpu_memory_utilization = GPU_MEMORY_UTILIZATION)

# ============================================================
# TODO — YOU WRITE THIS
#
# Task C: Generate outputs for `prompts`.
#
# Ask yourself:
#   1. Which object owns the generate() method?
#   2. What two main arguments does it need here?
#
# outputs = ...
# ============================================================

outputs = llm.generate(
    prompts,
    sampling_params
)


# ============================================================
# Inspect RequestOutput
# Small offline timing experiment:
# ============================================================

short_prompt = "Briefly explain GPU memory bandwidth."
long_prompt = ("Explain the relationship between GPU compute throughput, memory "
               "bandwidth, arithmetic intensity, and kernel performance. " * 80)

def run_once(prompt):
    # ========================================================
    # TODO — YOU WRITE THIS
    #
    # Measure wall-clock latency around ONE llm.generate call.
    # Return:
    #   latency_seconds, output_token_count
    #
    # Notes:
    # - This is NOT yet a rigorous serving benchmark.
    # - llm.generate is offline/batched inference, not HTTP TTFT.
    # - We only want a Day-1 baseline and familiarity with outputs.
    # ========================================================
    # raise NotImplementedError
    start_time = time.perf_counter()
    output = llm.generate(
        [prompt],
        sampling_params
    )
    end_time = time.perf_counter()
    latency_seconds = end_time - start_time
    output_token_count = len(output[0].outputs[0].token_ids)
    return latency_seconds, output_token_count

# Warm-up: complete this after run_once() works.
# _ = run_once(short_prompt)

# ============================================================
# TODO — YOU WRITE THIS
#
# Run each prompt 3 times and report:
#   mean latency
#   mean output-token count
#
# Compare short_prompt vs long_prompt.
# ============================================================
short_results = [run_once(short_prompt) for _ in range(3)]
long_results = [run_once(long_prompt) for _ in range(3)]

short_latencies = [x[0] for x in short_results]
short_tokens = [x[1] for x in short_results]

long_latencies = [x[0] for x in long_results]
long_tokens = [x[1] for x in long_results]

with open(output_path, "w", encoding="utf-8") as f:

    # ========================================================
    # RequestOutput inspection
    # ========================================================
    f.write("=== RequestOutput Inspection ===\n\n")

    for output in outputs:
        candidate = output.outputs[0]

        f.write(f"Request ID: {output.request_id}\n")
        f.write(f"Prompt: {output.prompt}\n")
        f.write(f"Prompt token count: {len(output.prompt_token_ids)}\n")
        f.write(f"Finished: {output.finished}\n")

        f.write(f"Generated text: {candidate.text}\n")
        f.write(f"Generated token IDs: {candidate.token_ids}\n")
        f.write(f"Generated token count: {len(candidate.token_ids)}\n")
        f.write(f"Finish reason: {candidate.finish_reason}\n")

        f.write("-" * 60 + "\n")

    # ========================================================
    # Latency benchmark
    # ========================================================
    f.write("\n=== Offline Latency Benchmark ===\n\n")

    f.write("Short prompt:\n")
    f.write(f"Mean latency: {statistics.mean(short_latencies):.4f} s\n")
    f.write(f"Mean output tokens: {statistics.mean(short_tokens):.2f}\n")
    f.write(f"Raw latencies: {short_latencies}\n")
    f.write("\n")

    f.write("Long prompt:\n")
    f.write(f"Mean latency: {statistics.mean(long_latencies):.4f} s\n")
    f.write(f"Mean output tokens: {statistics.mean(long_tokens):.2f}\n")
    f.write(f"Raw latencies: {long_latencies}\n")

Overwriting /content/test_vllm.py


In [27]:
!python /content/test_vllm.py

INFO 09-18 11:08:59 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 09-18 11:09:00 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-18 11:09:00 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-18 11:09:00 [model.py:2021] Using max model len 4096
INFO 09-18 11:09:00 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-18 11:09:00 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=14752) INFO 09-18 11:09:05 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dt


### What you should notice

The important conceptual call today is:

```text
Python program
    ↓
LLM.generate(...)
    ↓
vLLM engine
    ↓
scheduler / KV cache / model runner
    ↓
GPU
    ↓
RequestOutput
```

Today we treat the middle as a black box. Day 2 starts opening it.


## 4. Inspect `RequestOutput`

## 5. Small offline timing experiment:


### Checkpoint 1 — write down what happened

Do **not** over-interpret the numbers yet.

| Workload | Mean latency | Mean generated tokens |
|---|---:|---:|
| short prompt | | |
| long prompt | | |

Questions:

1. Did the longer prompt increase latency?
2. Why might that happen?
3. Is this number TTFT? Why or why not?
4. Which stage should be more affected by prompt length: prefill or decode?


## 6. Inspect GPU memory after model load

In [28]:

print(torch.cuda.memory_summary(abbreviated=True))
!nvidia-smi


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Requested memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------


### Checkpoint 2

Record:

- GPU memory used after model load:
- Approximate free memory:
- Why does vLLM reserve significant GPU memory beyond model weights?
- What future structure do you expect to occupy much of that memory?

Do not worry if you cannot fully explain the last two yet — that is Day 4.



## 7. Release the offline engine before starting the server

A single T4 cannot comfortably host two copies of the same vLLM model.
Delete the offline engine and clear Python's references before starting a separate server process.


In [30]:

import gc

# Keep this infrastructure code as-is.
# del outputs
# del llm
gc.collect()
torch.cuda.empty_cache()

!nvidia-smi


Fri Sep 18 11:11:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


## 8. Start the OpenAI-compatible vLLM server

The server uses the same model but exposes it over HTTP.

This cell is infrastructure, so it is provided for you.

If it fails because a CLI flag changed in your installed vLLM version, inspect:

```bash
vllm serve --help
```


In [31]:

import subprocess
import time
import os
import signal

server_log = open("/content/vllm_server.log", "w")

server = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--dtype", "float16",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

print("Server PID:", server.pid)
print("Log: /content/vllm_server.log")


Server PID: 15441
Log: /content/vllm_server.log


## 9. Wait until the server is healthy

In [32]:

import requests
import time

for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("Server is ready.")
            break
    except Exception:
        pass

    if i % 5 == 0:
        print(f"Waiting... ({i})")
    time.sleep(2)
else:
    print("Server did not become healthy. Inspect the log:")
    print(open("/content/vllm_server.log").read()[-8000:])


Waiting... (0)
Waiting... (5)
Waiting... (10)
Waiting... (15)
Waiting... (20)
Waiting... (25)
Waiting... (30)
Waiting... (35)
Waiting... (40)
Waiting... (45)
Waiting... (50)
Waiting... (55)
Server is ready.


## 10. Send your first HTTP request — second core exercise

In [33]:

from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="EMPTY",
)

# ============================================================
# TODO — YOU WRITE THIS
#
# Send ONE chat completion request.
#
# Requirements:
#   model=MODEL
#   one user message asking:
#       "What problem does PagedAttention solve?"
#   temperature=0.0
#   max_tokens=64
#
# response = client.chat.completions.create(...)
# ============================================================

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What problem does PagedAttention solve?"
        }
    ],
    temperature=0.0,
    max_tokens=64
)

# ============================================================
# TODO — YOU WRITE THIS
#
# Print only the generated assistant text.
# Explore `response` if you don't know the object layout.
# ============================================================
print(response.choices[0].message.content)

I'm sorry, but I couldn't find any information about a specific problem called "PagedAttention" that needs to be solved. It's possible that there might be some confusion or typo in the name.

Could you please provide more details or context about what kind of problem you're referring to? This will help me



## 11. Observe streaming and approximate TTFT

This is the first place today where you should distinguish:

- **TTFT**: request sent → first generated token/chunk arrives
- **E2E latency**: request sent → generation finishes

For Day 1, measuring Python streaming-chunk arrival time is sufficient.
It is not yet our final benchmark methodology.


In [34]:

import time

# ============================================================
# TODO — YOU WRITE THIS
#
# Send a streaming Chat Completions request.
#
# Measure:
#   start_time
#   first_nonempty_content_time
#   finish_time
#
# Compute:
#   approximate_TTFT = first_nonempty_content_time - start_time
#   E2E_latency      = finish_time - start_time
#
# Requirements:
#   prompt: "Explain continuous batching in about 100 words."
#   temperature=0.0
#   max_tokens=128
#   stream=True
#
# Important:
# Some initial stream chunks may contain metadata/role with no text.
# Count TTFT at the FIRST NON-EMPTY generated content chunk.
# ============================================================
start_time = time.perf_counter()
first_nonempty_content_time = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain continuous batching in about 100 words."
        }
    ],
    temperature=0.0,
    max_tokens=128,
    stream=True
)

generated_text = ""

for chunk in stream:
    content = chunk.choices[0].delta.content

    if content:
        if first_nonempty_content_time is None:
            first_nonempty_content_time = time.perf_counter()

        generated_text += content
        print(content, end="", flush=True)

finish_time = time.perf_counter()

approximate_TTFT = first_nonempty_content_time - start_time
E2E_latency = finish_time - start_time

print("\n")
print(f"Approximate TTFT: {approximate_TTFT:.4f} s")
print(f"E2E latency:      {E2E_latency:.4f} s")

Continuous Batching is an optimization technique used in machine learning and data processing pipelines to improve efficiency and reduce latency. It involves dividing the input data into smaller batches that can be processed concurrently without waiting for previous batches to complete.

In traditional batch processing, all data is loaded into memory at once before being processed. This approach works well when dealing with small datasets or when there's sufficient computational resources available. However, it becomes inefficient as the size of the dataset grows due to increased memory usage and slower performance bottlenecks.

Continuous Batching addresses these issues by breaking down large datasets into multiple smaller batches during each iteration of training or processing. Each batch

Approximate TTFT: 0.0887 s
E2E latency:      1.9703 s



### Checkpoint 3

Record:

- Approximate TTFT:
- End-to-end latency:
- Generated output tokens (if you counted them):
- Why is TTFT different from total latency?
- Which part of inference dominates TTFT for a long prompt?


## 12. Locate the installed vLLM source tree

In [35]:

import inspect
import os
import vllm

vllm_root = os.path.dirname(inspect.getfile(vllm))
print("Installed vLLM source:", vllm_root)

# We only LOCATE files today. Do not dive deeply into them yet.
!find "$vllm_root/v1" -maxdepth 3 -type f | grep -E "(engine|scheduler|kv_cache|model_runner)" | head -80


Installed vLLM source: /usr/local/lib/python3.13/dist-packages/vllm
/usr/local/lib/python3.13/dist-packages/vllm/v1/__pycache__/kv_cache_interface.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/__pycache__/kv_cache_layout.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/__pycache__/kv_cache_spec_registry.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/async_scheduler.py
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/__pycache__/kv_cache_manager.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/__pycache__/kv_cache_utils.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/__pycache__/kv_cache_coordinator.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/__pycache__/kv_cache_metrics.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/__pycache__/single_type_kv_cache_manager.cpython-


## 13. Day-1 source map — fill this yourself

Without trying to understand every line, locate likely files/classes for:

```text
User / API
    ↓
LLM / serving frontend
    ↓
Engine / EngineCore
    ↓
Scheduler
    ↓
KV-cache management
    ↓
Model runner
    ↓
GPU
```

Fill in:

| Layer | File / class you found | What you THINK it does |
|---|---|---|
| Offline frontend | | |
| Engine / core | | |
| Scheduler | | |
| KV cache | | |
| Model runner | | |

It is okay if some guesses are wrong. Day 2 will verify the request path.


## 14. Stop the server

In [36]:

# Infrastructure cleanup.
if server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server.kill()

server_log.close()
print("Server stopped.")


Server stopped.



# Day 1 completion checklist

You are done only when all of these are true:

- [ ] T4 detected and environment versions recorded.
- [ ] vLLM loaded the model successfully.
- [ ] You personally wrote `SamplingParams`.
- [ ] You personally instantiated `LLM`.
- [ ] You personally called `generate`.
- [ ] You explored and parsed `RequestOutput`.
- [ ] Short/long offline timing experiment completed.
- [ ] GPU memory after model load inspected.
- [ ] vLLM OpenAI-compatible server started successfully.
- [ ] You personally wrote one HTTP chat request.
- [ ] You personally measured approximate streaming TTFT and E2E latency.
- [ ] You located the installed source files for Engine, Scheduler, KV cache, and ModelRunner.
- [ ] You can draw the rough pipeline below from memory:

```text
request
  ↓
frontend / engine
  ↓
scheduler
  ↓
KV-cache management
  ↓
model runner
  ↓
GPU
  ↓
output
```

## What NOT to do today

Do not start modifying:

- scheduler policy
- block allocation
- chunked prefill
- PagedAttention kernels
- continuous batching internals

Those come after we trace the request lifecycle.
